# 🤟 Sign Language Translation — ML Notebook
**Datasets :** ASL Alphabet (870 imgs, 29 classes) · Indian Sign Language (42 k imgs, 35 classes)  
**Model    :** MobileNetV2 (Transfer Learning)  
**Author   :** ML Engineer Workflow  


## 1 · Setup & GPU Check


In [ ]:
import os, random, time, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

# ── Reproducibility ──────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch : {torch.__version__}')


## 2 · Dataset Paths


In [ ]:
BASE        = Path(r'r:/Projects/Sign Language')
ASL_DIR     = BASE / 'asl-alphabet-train'
ISL_DIR     = BASE / 'indian sign language' / 'Indian'
MODELS_DIR  = BASE / 'models'
MODELS_DIR.mkdir(exist_ok=True)

ASL_MODEL   = MODELS_DIR / 'asl_model.pth'
ISL_MODEL   = MODELS_DIR / 'isl_model.pth'

print('ASL dataset :', ASL_DIR)
print('ISL dataset :', ISL_DIR)
print('Models dir  :', MODELS_DIR)


## 3 · Exploratory Data Analysis (EDA)


### 3.1 Class Distribution — ASL


In [ ]:
asl_classes = sorted([d.name for d in ASL_DIR.iterdir() if d.is_dir()])
asl_counts  = {c: len(list((ASL_DIR / c).glob('*.jpg'))) for c in asl_classes}

fig, ax = plt.subplots(figsize=(16, 4))
bars = ax.bar(asl_counts.keys(), asl_counts.values(),
              color=plt.cm.plasma(np.linspace(0.2, 0.9, len(asl_classes))))
ax.set_title('ASL — Images per Class', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Class'); ax.set_ylabel('Image Count')
ax.set_facecolor('#0e0e1a'); fig.patch.set_facecolor('#0e0e1a')
ax.tick_params(colors='white'); ax.title.set_color('white')
ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
for spine in ax.spines.values(): spine.set_edgecolor('#333')
plt.tight_layout(); plt.savefig(BASE / 'asl_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total ASL images : {sum(asl_counts.values()):,}')
print(f'Classes          : {len(asl_classes)}')
print(f'Avg / class      : {np.mean(list(asl_counts.values())):.0f}')


### 3.2 Class Distribution — ISL


In [ ]:
isl_classes = sorted([d.name for d in ISL_DIR.iterdir() if d.is_dir()])
isl_counts  = {c: len(list((ISL_DIR / c).glob('*.jpg'))) for c in isl_classes}

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(isl_counts.keys(), isl_counts.values(),
       color=plt.cm.cool(np.linspace(0.1, 0.9, len(isl_classes))))
ax.set_title('ISL — Images per Class', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Class'); ax.set_ylabel('Image Count')
ax.set_facecolor('#0e0e1a'); fig.patch.set_facecolor('#0e0e1a')
ax.tick_params(colors='white'); ax.title.set_color('white')
ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
for spine in ax.spines.values(): spine.set_edgecolor('#333')
plt.tight_layout(); plt.savefig(BASE / 'isl_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total ISL images : {sum(isl_counts.values()):,}')
print(f'Classes          : {len(isl_classes)}')
print(f'Avg / class      : {np.mean(list(isl_counts.values())):.0f}')


### 3.3 Sample Images — ASL


In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(18, 9))
fig.patch.set_facecolor('#0e0e1a')
fig.suptitle('ASL Sample Images (one per class)', color='white', fontsize=14, fontweight='bold')
for ax, cls in zip(axes.flat, asl_classes):
    imgs = list((ASL_DIR / cls).glob('*.jpg'))
    if not imgs: ax.axis('off'); continue
    img = Image.open(random.choice(imgs)).resize((128, 128))
    ax.imshow(img); ax.set_title(cls, color='#a084ff', fontsize=9, fontweight='bold')
    ax.axis('off')
for ax in axes.flat[len(asl_classes):]: ax.axis('off')
plt.tight_layout(); plt.savefig(BASE / 'asl_samples.png', dpi=120, bbox_inches='tight')
plt.show()


### 3.4 Sample Images — ISL


In [ ]:
fig, axes = plt.subplots(5, 7, figsize=(18, 13))
fig.patch.set_facecolor('#0e0e1a')
fig.suptitle('ISL Sample Images (one per class)', color='white', fontsize=14, fontweight='bold')
for ax, cls in zip(axes.flat, isl_classes):
    imgs = list((ISL_DIR / cls).glob('*.jpg'))
    if not imgs: ax.axis('off'); continue
    img = Image.open(random.choice(imgs)).resize((128, 128))
    ax.imshow(img); ax.set_title(cls, color='#00d4ff', fontsize=9, fontweight='bold')
    ax.axis('off')
for ax in axes.flat[len(isl_classes):]: ax.axis('off')
plt.tight_layout(); plt.savefig(BASE / 'isl_samples.png', dpi=120, bbox_inches='tight')
plt.show()


### 3.5 Augmentation Preview


In [ ]:
aug = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.3, p=1.0),
])

sample_path = random.choice(list((ASL_DIR / 'A').glob('*.jpg')))
orig = Image.open(sample_path).resize((224, 224))

fig, axes = plt.subplots(1, 7, figsize=(18, 3))
fig.patch.set_facecolor('#0e0e1a')
fig.suptitle('Augmentation Preview — ASL class A', color='white', fontsize=12, fontweight='bold')
axes[0].imshow(orig); axes[0].set_title('Original', color='#a084ff', fontsize=9); axes[0].axis('off')
for i in range(1, 7):
    axes[i].imshow(aug(orig))
    axes[i].set_title(f'Aug #{i}', color='#00d4ff', fontsize=9)
    axes[i].axis('off')
plt.tight_layout(); plt.savefig(BASE / 'augmentation_preview.png', dpi=120, bbox_inches='tight')
plt.show()


## 4 · Model Architecture


In [ ]:
def build_model(num_classes, pretrained=True):
    weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
    model   = models.mobilenet_v2(weights=weights)
    for param in model.parameters(): param.requires_grad = False
    for layer in list(model.features.children())[-4:]:
        for param in layer.parameters(): param.requires_grad = True
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes),
    )
    return model

# Parameter count
demo = build_model(29)
total  = sum(p.numel() for p in demo.parameters())
trainable = sum(p.numel() for p in demo.parameters() if p.requires_grad)
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainable:,}  ({trainable/total*100:.1f}% of total)')


## 5 · Training Helpers


In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def get_transforms(heavy_aug=False):
    if heavy_aug:
        train_tf = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
            transforms.RandomGrayscale(p=0.1),
            transforms.RandomPerspective(0.3, p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    else:
        train_tf = transforms.Compose([
            transforms.Resize((240, 240)),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(0.2, 0.2, 0.1),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    val_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    return train_tf, val_tf

def make_loaders(data_dir, batch_size, val_split, heavy_aug=False):
    train_tf, val_tf = get_transforms(heavy_aug)
    full = datasets.ImageFolder(data_dir, transform=train_tf)
    n_val   = int(len(full) * val_split)
    n_train = len(full) - n_val
    train_set, val_set = random_split(full, [n_train, n_val],
                                      generator=torch.Generator().manual_seed(SEED))
    val_set.dataset = datasets.ImageFolder(data_dir, transform=val_tf)
    train_ldr = DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
    val_ldr   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    return train_ldr, val_ldr, full.classes

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = total_correct = total = 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if is_train: optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            if is_train: loss.backward(); optimizer.step()
            total_loss    += loss.item() * imgs.size(0)
            total_correct += (out.argmax(1) == labels).sum().item()
            total         += imgs.size(0)
    return total_loss / total, total_correct / total * 100

print('Helper functions defined ✓')


## 6 · Train ASL Model (A-Z + del, nothing, space — 29 classes)


In [ ]:
ASL_CFG = dict(batch_size=32, val_split=0.20, epochs=15, lr=1e-3)

train_ldr_asl, val_ldr_asl, asl_classes_list = make_loaders(
    ASL_DIR, ASL_CFG['batch_size'], ASL_CFG['val_split'], heavy_aug=True)
print(f'ASL → Train: {len(train_ldr_asl.dataset):,} | Val: {len(val_ldr_asl.dataset):,} | Classes: {len(asl_classes_list)}')


In [ ]:
asl_model = build_model(len(asl_classes_list)).to(DEVICE)
asl_crit  = nn.CrossEntropyLoss(label_smoothing=0.1)
asl_opt   = optim.Adam(filter(lambda p: p.requires_grad, asl_model.parameters()),
                        lr=ASL_CFG['lr'], weight_decay=1e-4)
asl_sched = optim.lr_scheduler.CosineAnnealingLR(asl_opt, T_max=ASL_CFG['epochs'])

asl_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_asl_acc = 0.0

for epoch in range(1, ASL_CFG['epochs'] + 1):
    t0 = time.time()
    tl, ta = run_epoch(asl_model, train_ldr_asl, asl_crit, asl_opt)
    vl, va = run_epoch(asl_model, val_ldr_asl,   asl_crit)
    asl_sched.step()
    asl_history['train_loss'].append(tl); asl_history['train_acc'].append(ta)
    asl_history['val_loss'].append(vl);   asl_history['val_acc'].append(va)
    tag = ''
    if va > best_asl_acc:
        best_asl_acc = va
        torch.save({'model_state_dict': asl_model.state_dict(),
                    'class_names': asl_classes_list, 'num_classes': len(asl_classes_list),
                    'val_acc': va, 'epoch': epoch}, ASL_MODEL)
        tag = '  ← best ✓'
    print(f'[ASL] Ep {epoch:02}/{ASL_CFG["epochs"]} | '
          f'TrLoss {tl:.4f} TrAcc {ta:.1f}% | '
          f'VaLoss {vl:.4f} VaAcc {va:.1f}% | '
          f'{time.time()-t0:.0f}s{tag}')

print(f'\nASL best val accuracy: {best_asl_acc:.1f}%')


### 6.1 ASL Training Curves


In [ ]:
epochs_range = range(1, ASL_CFG['epochs'] + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0e0e1a')
for ax in (ax1, ax2):
    ax.set_facecolor('#0e0e1a')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for s in ax.spines.values(): s.set_edgecolor('#333')

ax1.plot(epochs_range, asl_history['train_loss'], '#a084ff', lw=2, label='Train')
ax1.plot(epochs_range, asl_history['val_loss'],   '#00d4ff', lw=2, label='Val', linestyle='--')
ax1.set_title('ASL Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(facecolor='#1a1a2e', labelcolor='white')

ax2.plot(epochs_range, asl_history['train_acc'], '#a084ff', lw=2, label='Train')
ax2.plot(epochs_range, asl_history['val_acc'],   '#00d4ff', lw=2, label='Val', linestyle='--')
ax2.axhline(best_asl_acc, color='#22d67a', linestyle=':', lw=1.5, label=f'Best {best_asl_acc:.1f}%')
ax2.set_title('ASL Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(facecolor='#1a1a2e', labelcolor='white')

plt.tight_layout(); plt.savefig(BASE / 'asl_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 7 · Train ISL Model (A-Z + 1-9 — 35 classes)


In [ ]:
ISL_CFG = dict(batch_size=64, val_split=0.15, epochs=10, lr=1e-3)

train_ldr_isl, val_ldr_isl, isl_classes_list = make_loaders(
    ISL_DIR, ISL_CFG['batch_size'], ISL_CFG['val_split'], heavy_aug=False)
print(f'ISL → Train: {len(train_ldr_isl.dataset):,} | Val: {len(val_ldr_isl.dataset):,} | Classes: {len(isl_classes_list)}')


In [ ]:
isl_model = build_model(len(isl_classes_list)).to(DEVICE)
isl_crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
isl_opt   = optim.Adam(filter(lambda p: p.requires_grad, isl_model.parameters()),
                        lr=ISL_CFG['lr'], weight_decay=1e-4)
isl_sched = optim.lr_scheduler.OneCycleLR(isl_opt, max_lr=ISL_CFG['lr'],
                                           steps_per_epoch=len(train_ldr_isl),
                                           epochs=ISL_CFG['epochs'])

isl_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_isl_acc = 0.0

for epoch in range(1, ISL_CFG['epochs'] + 1):
    t0 = time.time()
    # train with scheduler step per batch
    isl_model.train()
    ep_loss = ep_correct = ep_total = 0
    for imgs, labels in train_ldr_isl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        isl_opt.zero_grad()
        out  = isl_model(imgs)
        loss = isl_crit(out, labels)
        loss.backward(); isl_opt.step(); isl_sched.step()
        ep_loss    += loss.item() * imgs.size(0)
        ep_correct += (out.argmax(1) == labels).sum().item()
        ep_total   += imgs.size(0)
    tl, ta = ep_loss / ep_total, ep_correct / ep_total * 100
    vl, va = run_epoch(isl_model, val_ldr_isl, isl_crit)
    isl_history['train_loss'].append(tl); isl_history['train_acc'].append(ta)
    isl_history['val_loss'].append(vl);   isl_history['val_acc'].append(va)
    tag = ''
    if va > best_isl_acc:
        best_isl_acc = va
        torch.save({'model_state_dict': isl_model.state_dict(),
                    'class_names': isl_classes_list, 'num_classes': len(isl_classes_list),
                    'val_acc': va, 'epoch': epoch}, ISL_MODEL)
        tag = '  ← best ✓'
    print(f'[ISL] Ep {epoch:02}/{ISL_CFG["epochs"]} | '
          f'TrLoss {tl:.4f} TrAcc {ta:.1f}% | '
          f'VaLoss {vl:.4f} VaAcc {va:.1f}% | '
          f'{time.time()-t0:.0f}s{tag}')

print(f'\nISL best val accuracy: {best_isl_acc:.1f}%')


### 7.1 ISL Training Curves


In [ ]:
epochs_range_isl = range(1, ISL_CFG['epochs'] + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0e0e1a')
for ax in (ax1, ax2):
    ax.set_facecolor('#0e0e1a'); ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for s in ax.spines.values(): s.set_edgecolor('#333')

ax1.plot(epochs_range_isl, isl_history['train_loss'], '#00d4ff', lw=2, label='Train')
ax1.plot(epochs_range_isl, isl_history['val_loss'],   '#a084ff', lw=2, label='Val', linestyle='--')
ax1.set_title('ISL Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(facecolor='#1a1a2e', labelcolor='white')

ax2.plot(epochs_range_isl, isl_history['train_acc'], '#00d4ff', lw=2, label='Train')
ax2.plot(epochs_range_isl, isl_history['val_acc'],   '#a084ff', lw=2, label='Val', linestyle='--')
ax2.axhline(best_isl_acc, color='#22d67a', linestyle=':', lw=1.5, label=f'Best {best_isl_acc:.1f}%')
ax2.set_title('ISL Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(facecolor='#1a1a2e', labelcolor='white')

plt.tight_layout(); plt.savefig(BASE / 'isl_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 8 · Model Evaluation


### 8.1 ASL Confusion Matrix


In [ ]:
def get_preds(model, loader, class_names):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            preds = model(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

y_true_asl, y_pred_asl = get_preds(asl_model, val_ldr_asl, asl_classes_list)
cm_asl = confusion_matrix(y_true_asl, y_pred_asl)

fig, ax = plt.subplots(figsize=(14, 12))
fig.patch.set_facecolor('#0e0e1a'); ax.set_facecolor('#0e0e1a')
sns.heatmap(cm_asl, annot=True, fmt='d', cmap='Purples',
            xticklabels=asl_classes_list, yticklabels=asl_classes_list,
            linewidths=0.5, linecolor='#1a1a2e', ax=ax)
ax.set_title('ASL Confusion Matrix (Validation Set)', color='white', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Predicted', color='white'); ax.set_ylabel('True', color='white')
ax.tick_params(colors='white')
plt.tight_layout(); plt.savefig(BASE / 'asl_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(classification_report(y_true_asl, y_pred_asl, target_names=asl_classes_list))


### 8.2 ISL Confusion Matrix


In [ ]:
y_true_isl, y_pred_isl = get_preds(isl_model, val_ldr_isl, isl_classes_list)
cm_isl = confusion_matrix(y_true_isl, y_pred_isl)

fig, ax = plt.subplots(figsize=(16, 14))
fig.patch.set_facecolor('#0e0e1a'); ax.set_facecolor('#0e0e1a')
sns.heatmap(cm_isl, annot=True, fmt='d', cmap='Blues',
            xticklabels=isl_classes_list, yticklabels=isl_classes_list,
            linewidths=0.5, linecolor='#1a1a2e', ax=ax)
ax.set_title('ISL Confusion Matrix (Validation Set)', color='white', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Predicted', color='white'); ax.set_ylabel('True', color='white')
ax.tick_params(colors='white')
plt.tight_layout(); plt.savefig(BASE / 'isl_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(classification_report(y_true_isl, y_pred_isl, target_names=isl_classes_list))


### 8.3 Per-Class Accuracy Comparison


In [ ]:
asl_per_class = cm_asl.diagonal() / cm_asl.sum(axis=1) * 100

fig, ax = plt.subplots(figsize=(16, 5))
fig.patch.set_facecolor('#0e0e1a'); ax.set_facecolor('#0e0e1a')
colors = ['#22d67a' if v >= 90 else '#ff9240' if v >= 70 else '#ff4d6d' for v in asl_per_class]
ax.bar(asl_classes_list, asl_per_class, color=colors)
ax.axhline(90, color='#22d67a', linestyle='--', lw=1, alpha=0.5, label='90% line')
ax.set_ylim(0, 110); ax.set_title('ASL Per-Class Accuracy', color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Class', color='white'); ax.set_ylabel('Accuracy (%)', color='white')
ax.tick_params(colors='white')
for s in ax.spines.values(): s.set_edgecolor('#333')
green_p = mpatches.Patch(color='#22d67a', label='≥ 90%')
orange_p = mpatches.Patch(color='#ff9240', label='70–90%')
red_p = mpatches.Patch(color='#ff4d6d', label='< 70%')
ax.legend(handles=[green_p, orange_p, red_p], facecolor='#1a1a2e', labelcolor='white')
plt.tight_layout(); plt.savefig(BASE / 'asl_per_class_acc.png', dpi=150, bbox_inches='tight')
plt.show()


## 9 · Inference Demo


In [ ]:
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def predict_single(model, class_names, img_path):
    img    = Image.open(img_path).convert('RGB')
    tensor = val_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    top3_p, top3_i = torch.topk(probs, 3)
    return img, [(class_names[i], p.item()) for i, p in zip(top3_i, top3_p)]

# Pick 12 random ASL val samples
sample_paths, sample_truths = [], []
for cls in random.sample(asl_classes_list, min(12, len(asl_classes_list))):
    imgs_in_cls = list((ASL_DIR / cls).glob('*.jpg'))
    if imgs_in_cls:
        sample_paths.append(random.choice(imgs_in_cls))
        sample_truths.append(cls)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.patch.set_facecolor('#0e0e1a')
fig.suptitle('ASL Inference Demo', color='white', fontsize=14, fontweight='bold')

for ax, path, true_cls in zip(axes.flat, sample_paths, sample_truths):
    img, top3 = predict_single(asl_model, asl_classes_list, path)
    pred_cls, pred_conf = top3[0]
    correct = pred_cls == true_cls
    ax.imshow(img)
    color = '#22d67a' if correct else '#ff4d6d'
    ax.set_title(f'True: {true_cls}  |  Pred: {pred_cls} ({pred_conf*100:.0f}%)',
                 color=color, fontsize=9, fontweight='bold')
    for s in ax.spines.values(): s.set_edgecolor(color); s.set_linewidth(3)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.tight_layout(); plt.savefig(BASE / 'asl_inference_demo.png', dpi=120, bbox_inches='tight')
plt.show()


## 10 · Final Summary


In [ ]:
print('='*55)
print('  SIGN LANGUAGE TRANSLATION — TRAINING SUMMARY')
print('='*55)
print(f'  ASL Model : {len(asl_classes_list)} classes | Best Val Acc = {best_asl_acc:.1f}%')
print(f'  ISL Model : {len(isl_classes_list)} classes | Best Val Acc = {best_isl_acc:.1f}%')
print(f'  ASL saved : {ASL_MODEL}')
print(f'  ISL saved : {ISL_MODEL}')
print('='*55)
print()
print('Next steps:')
print('  1. python app.py          → Start Flask API')
print('  2. Open web/index.html    → Launch UI in browser')
print('  3. Sign in front of cam   → See real-time predictions!')
